In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [ ]:
path = r'C:\Data_analysis\Thesis\Data\LF_data_Trimed.csv'
df = pd.read_csv(path,sep=',')
df.head(5)

In [ ]:
spike_cols = [
    "PV_WS_Radiation", "PV_WS_AirTemp", "PV_WS_RelHum",
    "BA_Soc","BA_TotActPwr_BESS_AC_Panel2","BA_TotActPwr_BESS_AC_Panel1",
    "BU_TotActPwr_Tech_Room","BU_TotActPwr_SDB_EL_Substation",
    "BU_TotActPwr_Academy","BU_PwrReq",
    "EL_TotActPwr_NitrogenUnit",
]

In [ ]:
GROUPS = {
    "Building_load": [
        "BU_TotActPwr_Tech_Room",
        "ELX_TotActPwr_Pump_Room",
        "BU_TotActPwr_SDB_EL_Substation",
        "BU_TotActPwr_Academy",
        "BU_PwrReq",
        "BU_Unitstate"

    ],
    "pv_weather": [
        "PV_WS_Radiation",
        "PV_WS_AirTemp",
        "PV_WS_RelHum",
        "PV_WS_RelAirPre",
        "PV_Unitstate"
    ],
    "BESS": [
        "BA_PwrAtChrLimTot",
        "BA_PwrAtDisLimTot",
        "BA_TotActPwr_BESS_AC_Panel2",
        "BA_TotActPwr_BESS_AC_Panel1",
        "BA_PwrAt",
        "BA_Unitstate",
        "BA_Soc"
    ],
    "ELX": [
        'EL_AuxPwr', 'EL_TotActPwr_NitrogenUnit'
    ],
}

### BESS pipeline


In [ ]:
def ensure_datetime_index(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d.index = pd.to_datetime(d.index, errors="coerce")
    d = d[~d.index.isna()].sort_index()
    d = d[~d.index.duplicated(keep="last")]
    return d

def keep_long_runs(mask: pd.Series, min_samples: int) -> pd.Series:
    m = mask.fillna(False).astype(bool)
    g = (m != m.shift()).cumsum()
    out = pd.Series(False, index=m.index)
    for _, grp in m.groupby(g):
        if grp.iloc[0] and len(grp) >= min_samples:
            out.loc[grp.index] = True
    return out

def run_intervals(mask: pd.Series) -> pd.DataFrame:
    m = mask.fillna(False).astype(bool)
    g = (m != m.shift()).cumsum()
    rows = []
    for _, grp in m.groupby(g):
        if grp.iloc[0]:
            rows.append((grp.index[0], grp.index[-1], len(grp)))
    return pd.DataFrame(rows, columns=["start", "end", "samples"])

def merge_intervals(intervals: pd.DataFrame, max_gap: pd.Timedelta) -> pd.DataFrame:
    if intervals is None or intervals.empty:
        return pd.DataFrame(columns=["start", "end"])
    ints = intervals.sort_values("start").reset_index(drop=True)
    merged = []
    cur_start = ints.loc[0, "start"]
    cur_end = ints.loc[0, "end"]
    for i in range(1, len(ints)):
        s = ints.loc[i, "start"]
        e = ints.loc[i, "end"]
        if (s - cur_end) <= max_gap:
            cur_end = max(cur_end, e)
        else:
            merged.append((cur_start, cur_end))
            cur_start, cur_end = s, e
    merged.append((cur_start, cur_end))
    return pd.DataFrame(merged, columns=["start", "end"])

def detect_bess_limit_intervals(
    df: pd.DataFrame,
    *,
    freq: str = "5min",
    tol_zero_limits: float = 0.0,
    min_minutes: int = 30,
    chr_col: str = "BA_PwrAtChrLimTot",
    dis_col: str = "BA_PwrAtDisLimTot",
):
    d = ensure_datetime_index(df)

    for c in (chr_col, dis_col):
        if c not in d.columns:
            raise KeyError(f"Missing required column: {c}")

    chr0 = pd.to_numeric(d[chr_col], errors="coerce") <= tol_zero_limits
    dis0 = pd.to_numeric(d[dis_col], errors="coerce") <= tol_zero_limits
    base_mask = chr0 & dis0

    step = pd.to_timedelta(freq)
    min_samples = int(np.ceil(pd.Timedelta(minutes=min_minutes) / step))

    event_mask = keep_long_runs(base_mask, min_samples=min_samples)
    intervals_raw = run_intervals(event_mask)

    debug = {
        "total_rows": int(len(d)),
        "base_hits": int(base_mask.sum()),
        "event_hits_after_duration_filter": int(event_mask.sum()),
        "num_intervals": int(len(intervals_raw)),
        "min_samples": int(min_samples),
        "freq": freq,
        "tol_zero_limits": tol_zero_limits,
        "min_minutes": min_minutes,
    }
    return d, event_mask, intervals_raw, debug

In [ ]:
def stuck_decision_graded(
    seg: pd.Series,
    *,
    step: pd.Timedelta,
    # thresholds
    zero_tol: float = 0.0,
    high_thr: float = 0.95,
    med_thr: float = 0.85,
    # duration gates
    probable_min_hours: float = 2.0,
    change_rate_thr: float = 0.01,
    change_min_hours: float = 1.0,
    # robustness
    min_non_nan: int = 10,
):
    """
    Returns:
      stuck_level: "definite" | "probable" | "no"
      reason: str
      metrics: dict

    Metrics computed on non-NaN values:
      zero_ratio, dominant_ratio, change_rate, iqr, mad
    """

    x = pd.to_numeric(seg, errors="coerce")
    non_nan = x.dropna()

    samples = int(len(seg))
    non_nan_n = int(len(non_nan))

    # duration of this interval by sample count (aligned with your run logic)
    duration = samples * step
    duration_hours = float(duration / pd.Timedelta(hours=1))

    # Default metrics
    metrics = {
        "samples": samples,
        "non_nan": non_nan_n,
        "duration": duration,
        "duration_hours": duration_hours,
        "zero_ratio": np.nan,
        "dominant_ratio": np.nan,
        "change_rate": np.nan,
        "iqr": np.nan,
        "mad": np.nan,
    }

    if non_nan_n < min_non_nan:
        # Too little reliable data -> treat as definite stuck (safe cleaning)
        return "definite", "too_few_valid_samples", metrics

    # Ratios
    zero_ratio = float((non_nan.abs() <= zero_tol).mean())
    vc = non_nan.value_counts(normalize=True)
    dominant_ratio = float(vc.iloc[0]) if len(vc) else 0.0

    # Activity: how often it changes (ignores NaNs)
    changes = (non_nan != non_nan.shift()).sum()
    change_rate = float(changes / max(non_nan_n, 1))

    # Robust variation
    q75 = float(non_nan.quantile(0.75))
    q25 = float(non_nan.quantile(0.25))
    iqr = q75 - q25
    med = float(non_nan.median())
    mad = float((non_nan - med).abs().median())

    metrics.update({
        "zero_ratio": zero_ratio,
        "dominant_ratio": dominant_ratio,
        "change_rate": change_rate,
        "iqr": iqr,
        "mad": mad,
    })

    # --- Decision rules ---
    # 1) Definite stuck
    if (zero_ratio >= high_thr) or (dominant_ratio >= high_thr):
        return "definite", "high_ratio_stuck", metrics

    # 2) Probable stuck (needs duration)
    if ((zero_ratio >= med_thr) or (dominant_ratio >= med_thr)) and (duration_hours >= probable_min_hours):
        return "probable", "medium_ratio_long_interval", metrics

    # 3) Probable stuck (very low activity for long enough)
    if (change_rate <= change_rate_thr) and (duration_hours >= change_min_hours):
        return "probable", "low_activity_long_interval", metrics

    return "no", "dynamic", metrics

In [ ]:
def bess_pipeline_graded_nanify_then_merge(
    df: pd.DataFrame,
    *,
    bess_cols: list[str],
    freq: str = "5min",
    tol_zero_limits: float = 0.0,
    min_minutes: int = 30,
    merge_gap_minutes: int = 30,
    # graded stuck params
    zero_tol_signals: float = 0.0,
    high_thr: float = 0.95,
    med_thr: float = 0.85,
    probable_min_hours: float = 2.0,
    change_rate_thr: float = 0.01,
    change_min_hours: float = 1.0,
    min_non_nan: int = 10,
    # what to NaN
    nanify_levels: tuple[str, ...] = ("definite", "probable"),
    # limit columns
    chr_col: str = "BA_PwrAtChrLimTot",
    dis_col: str = "BA_PwrAtDisLimTot",
):
    """
    Returns:
      cleaned_df
      intervals_raw
      intervals_merged
      report (with stuck_level + metrics)
      debug
    """
    # --- detect raw intervals ---
    d, event_mask, intervals_raw, debug1 = detect_bess_limit_intervals(
        df,
        freq=freq,
        tol_zero_limits=tol_zero_limits,
        min_minutes=min_minutes,
        chr_col=chr_col,
        dis_col=dis_col,
    )

    step = pd.to_timedelta(freq)
    cleaned = d.copy()
    cols = [c for c in bess_cols if c in cleaned.columns]

    rows = []
    for interval_id, r in intervals_raw.reset_index(drop=True).iterrows():
        a, b = r["start"], r["end"]

        for col in cols:
            seg = cleaned.loc[a:b, col]

            stuck_level, reason, metrics = stuck_decision_graded(
                seg,
                step=step,
                zero_tol=zero_tol_signals,
                high_thr=high_thr,
                med_thr=med_thr,
                probable_min_hours=probable_min_hours,
                change_rate_thr=change_rate_thr,
                change_min_hours=change_min_hours,
                min_non_nan=min_non_nan,
            )

            rows.append({
                "interval_id": int(interval_id),
                "start": a,
                "end": b,
                "column": col,
                "stuck_level": stuck_level,  # definite / probable / no
                "reason": reason,
                **metrics,
            })

            if stuck_level in nanify_levels:
                cleaned.loc[a:b, col] = np.nan

    report = pd.DataFrame(rows)

    # --- merge only for reporting ---
    intervals_merged = merge_intervals(intervals_raw, max_gap=pd.Timedelta(minutes=merge_gap_minutes))

    debug = {
        **debug1,
        "merge_gap_minutes": int(merge_gap_minutes),
        "num_intervals_merged": int(len(intervals_merged)),
        "bess_cols_checked": int(len(cols)),
        "nanify_levels": list(nanify_levels),
        "graded_thresholds": {
            "zero_tol_signals": zero_tol_signals,
            "high_thr": high_thr,
            "med_thr": med_thr,
            "probable_min_hours": probable_min_hours,
            "change_rate_thr": change_rate_thr,
            "change_min_hours": change_min_hours,
            "min_non_nan": min_non_nan,
        }
    }

    return cleaned, intervals_raw, intervals_merged, report, debug

In [ ]:
def detect_bess_limit_intervals(
    df: pd.DataFrame,
    *,
    freq: str = "5min",
    tol_zero_limits: float = 0.0,
    min_minutes: int = 30,
    chr_col: str = "BA_PwrAtChrLimTot",
    dis_col: str = "BA_PwrAtDisLimTot",
):
    d = ensure_datetime_index(df)

    for c in (chr_col, dis_col):
        if c not in d.columns:
            raise KeyError(f"Missing required column: {c}")

    chr0 = pd.to_numeric(d[chr_col], errors="coerce") <= tol_zero_limits
    dis0 = pd.to_numeric(d[dis_col], errors="coerce") <= tol_zero_limits
    base_mask = chr0 & dis0

    step = pd.to_timedelta(freq)
    min_samples = int(np.ceil(pd.Timedelta(minutes=min_minutes) / step))

    event_mask = keep_long_runs(base_mask, min_samples=min_samples)
    intervals_raw = run_intervals(event_mask)

    debug = {
        "total_rows": int(len(d)),
        "base_hits": int(base_mask.sum()),
        "event_hits_after_duration_filter": int(event_mask.sum()),
        "num_intervals": int(len(intervals_raw)),
        "min_samples": int(min_samples),
        "freq": freq,
        "tol_zero_limits": tol_zero_limits,
        "min_minutes": min_minutes,
    }
    return d, event_mask, intervals_raw, debug

In [ ]:
print("detect_bess_limit_intervals" in globals())
print([k for k in globals().keys() if "detect" in k and "bess" in k])

In [ ]:
cleaned_bess_df, intervals_raw, intervals_merged, report, debug = (
    bess_pipeline_graded_nanify_then_merge(
        df,
        bess_cols=GROUPS["BESS"],
        freq="5min",
        tol_zero_limits=0.0,
        min_minutes=30,
        merge_gap_minutes=30,
        # graded stuck
        zero_tol_signals=0.0,
        high_thr=0.95,
        med_thr=0.85,
        probable_min_hours=2.0,
        change_rate_thr=0.01,
        change_min_hours=1.0,
        min_non_nan=10,
        # NaN both definite & probable (as you prefer)
        nanify_levels=("definite", "probable"),
    )
)

print(debug)
print("RAW intervals:\n", intervals_raw.to_string(index=False))
print("MERGED intervals:\n", intervals_merged.to_string(index=False))

# What got NaNified?
print(
    report[report["stuck_level"].isin(["definite", "probable"])]
    .loc[:, ["interval_id","column","stuck_level","reason","zero_ratio","dominant_ratio","change_rate","duration_hours"]]
    .sort_values(["interval_id","column"])
    .to_string(index=False))

In [ ]:
df2 = ensure_datetime_index(df)
common_idx = df2.index.intersection(cleaned_bess_df.index)
df2.loc[common_idx, GROUPS["BESS"]] = cleaned_bess_df.loc[common_idx, GROUPS["BESS"]]
df = df2

In [ ]:
def detect_pv_weather_zero_events(
    df: pd.DataFrame,
    *,
    pv_cols: list[str],
    freq: str = "5min",
    min_minutes: int = 30,
    zero_tol: float = 0.0,
    # how strict
    min_zero_ratio_per_col: float = 1.0,  # 1.0 means every sample in interval must be zero for each col (strict)
    min_cols_fraction: float = 0.75,      # e.g. 0.75 => at least 3/4 sensors must be zero at same timestamp
):
    """
    Detect PV-weather communication loss/maintenance events when most PV weather sensors are zero.

    Event mask at timestamp t:
      fraction_of_cols_zero(t) >= min_cols_fraction

    Then apply duration filter: event must last >= min_minutes.

    Returns: d, event_mask, intervals_raw, debug
    """
    d = ensure_datetime_index(df)

    cols = [c for c in pv_cols if c in d.columns]
    if not cols:
        raise KeyError("None of the pv_cols exist in df.")

    step = pd.to_timedelta(freq)
    min_samples = int(np.ceil(pd.Timedelta(minutes=min_minutes) / step))

    # zero mask per column at each timestamp
    Z = pd.DataFrame(index=d.index)
    for c in cols:
        x = pd.to_numeric(d[c], errors="coerce")
        Z[c] = (x.abs() <= zero_tol)

    # timestamp-level event: enough columns are zero
    frac_zero = Z.mean(axis=1)
    base_mask = frac_zero >= min_cols_fraction

    # duration filter
    event_mask = keep_long_runs(base_mask, min_samples=min_samples)
    intervals_raw = run_intervals(event_mask)

    debug = {
        "total_rows": int(len(d)),
        "pv_cols_used": cols,
        "min_cols_fraction": float(min_cols_fraction),
        "min_minutes": int(min_minutes),
        "freq": freq,
        "min_samples": int(min_samples),
        "base_hits": int(base_mask.sum()),
        "event_hits_after_duration_filter": int(event_mask.sum()),
        "num_intervals": int(len(intervals_raw)),
    }
    return d, event_mask, intervals_raw, debug


def pv_weather_nanify_pipeline(
    df: pd.DataFrame,
    *,
    pv_weather_cols: list[str],          # sensors to NaN (radiation/temp/hum/pressure)
    freq: str = "5min",
    min_minutes: int = 30,
    zero_tol: float = 0.0,
    min_cols_fraction: float = 0.75,
    merge_gap_minutes: int = 30,
):
    """
    Full PV-weather pipeline:
      1) detect zero-based comm-loss intervals (timestamp-level)
      2) merge short gaps for reporting
      3) set PV weather sensor columns to NaN inside event intervals

    Returns:
      cleaned_df, intervals_raw, intervals_merged, debug
    """
    d, event_mask, intervals_raw, debug1 = detect_pv_weather_zero_events(
        df,
        pv_cols=pv_weather_cols,
        freq=freq,
        min_minutes=min_minutes,
        zero_tol=zero_tol,
        min_cols_fraction=min_cols_fraction,
    )

    cleaned = d.copy()

    # Apply NaNs during each raw interval
    for _, r in intervals_raw.iterrows():
        a, b = r["start"], r["end"]
        cleaned.loc[a:b, pv_weather_cols] = np.nan

    # Merge for reporting
    intervals_merged = merge_intervals(intervals_raw, max_gap=pd.Timedelta(minutes=merge_gap_minutes))

    debug = {
        **debug1,
        "merge_gap_minutes": int(merge_gap_minutes),
        "num_intervals_merged": int(len(intervals_merged)),
        "nanified_cols": pv_weather_cols,
        "zero_tol": float(zero_tol),
    }

    return cleaned, intervals_raw, intervals_merged, debug

In [ ]:
pv_sensor_cols = [
    "PV_WS_Radiation",
    "PV_WS_AirTemp",
    "PV_WS_RelHum",
    "PV_WS_RelAirPre",
]
# keep PV_Unitstate as-is (do not NaN)
cleaned_pv_df, pv_intervals_raw, pv_intervals_merged, pv_debug = pv_weather_nanify_pipeline(
    df,
    pv_weather_cols=pv_sensor_cols,
    freq="5min",
    min_minutes=30,
    zero_tol=0.0,
    min_cols_fraction=0.75,  # at least 3/4 sensors zero simultaneously
    merge_gap_minutes=30
)

print(pv_debug)
print("PV RAW intervals:\n", pv_intervals_raw.to_string(index=False))
print("PV MERGED intervals:\n", pv_intervals_merged.to_string(index=False))

In [ ]:
def evaluate_group_during_events(
    df: pd.DataFrame,
    *,
    event_mask: pd.Series,
    group_cols: list[str],
    zero_tol: float = 0.0,
    high_thr: float = 0.95,
    change_rate_thr: float = 0.01,
):
    """
    Evaluate whether signals become mostly-zero or frozen
    during system event windows.
    """

    d = ensure_datetime_index(df)
    event_mask = event_mask.reindex(d.index).fillna(False)

    results = []

    for col in group_cols:
        if col not in d.columns:
            continue

        x = pd.to_numeric(d[col], errors="coerce")
        seg = x[event_mask]
        seg = seg.dropna()

        if len(seg) < 10:
            continue

        zero_ratio = float((seg.abs() <= zero_tol).mean())
        vc = seg.value_counts(normalize=True)
        dominant_ratio = float(vc.iloc[0]) if len(vc) else 0.0
        changes = (seg != seg.shift()).sum()
        change_rate = float(changes / len(seg))

        # classification
        if zero_ratio >= high_thr:
            status = "mostly_zero"
        elif dominant_ratio >= high_thr:
            status = "mostly_constant"
        elif change_rate <= change_rate_thr:
            status = "low_activity"
        else:
            status = "dynamic"

        results.append({
            "column": col,
            "samples_in_event": len(seg),
            "zero_ratio": zero_ratio,
            "dominant_ratio": dominant_ratio,
            "change_rate": change_rate,
            "status": status,
        })

    return pd.DataFrame(results)


In [ ]:
def mask_from_intervals(index, intervals_df):
    m = pd.Series(False, index=index)
    if intervals_df is None or len(intervals_df) == 0:
        return m
    for _, r in intervals_df.iterrows():
        m.loc[r["start"]:r["end"]] = True
    return m

d = ensure_datetime_index(df)

bess_event_mask = mask_from_intervals(d.index, intervals_raw)
pv_event_mask   = mask_from_intervals(d.index, pv_intervals_raw)

system_event_mask = bess_event_mask | pv_event_mask
print("BESS event points:", int(bess_event_mask.sum()))
print("PV event points:", int(pv_event_mask.sum()))
print("System event points:", int(system_event_mask.sum()))

In [ ]:
building_report = evaluate_group_during_events(
    df,
    event_mask=system_event_mask,
    group_cols=GROUPS["Building_load"],
    zero_tol=0.0,
    high_thr=0.95,
    change_rate_thr=0.01,
)

elx_report = evaluate_group_during_events(
    df,
    event_mask=system_event_mask,
    group_cols=GROUPS["ELX"],
    zero_tol=0.0,
    high_thr=0.95,
    change_rate_thr=0.01,
)

print("Building during BESS/PV events:")
print(building_report.sort_values(["status","column"]).to_string(index=False))

print("\nELX during BESS/PV events:")
print(elx_report.sort_values(["status","column"]).to_string(index=False))

In [ ]:
system_event_mask = bess_event_mask | pv_event_mask

building_report = evaluate_group_during_events(
    df,
    event_mask=system_event_mask,
    group_cols=GROUPS["Building_load"],
)

elx_report = evaluate_group_during_events(
    df,
    event_mask=system_event_mask,
    group_cols=GROUPS["ELX"],
)

print("Building during events:")
print(building_report.to_string(index=False))

print("\nELX during events:")
print(elx_report.to_string(index=False))

In [ ]:
def compare_event_vs_normal(df, col, event_mask, zero_tol=0.0):
    x = pd.to_numeric(df[col], errors="coerce")

    normal = x[~event_mask]
    event  = x[event_mask]

    zr_normal = (normal.abs() <= zero_tol).mean()
    zr_event  = (event.abs() <= zero_tol).mean()

    return zr_normal, zr_event

In [ ]:
for col in ["ELX_TotActPwr_Pump_Room",
            "EL_AuxPwr",
            "EL_TotActPwr_NitrogenUnit"]:

    zr_n, zr_e = compare_event_vs_normal(df, col, system_event_mask)
    print(col, "normal:", round(zr_n,3), "event:", round(zr_e,3))

### Hampel outlier detection 

In [ ]:
# def hampel_filter_series(
#     s: pd.Series,
#     *,
#     window: int,
#     k: float,
#     center: bool = True,
#     min_periods: int | None = None,
#     mad_floor: float = 1e-9,
# ) -> tuple[pd.Series, pd.Series, pd.Series, pd.Series]:
#     """
#     Hampel filter on a single Series.

#     Returns:
#       is_outlier (bool Series)
#       rolling_median (Series)
#       rolling_mad (Series)
#       score (Series) = |x - median| / MAD

#     Notes:
#       - window should be odd ideally (e.g., 7, 9, 11)
#       - MAD floor prevents division by zero in very flat windows
#     """
#     x = pd.to_numeric(s, errors="coerce")

#     if min_periods is None:
#         # robust default: require at least half the window to compute stable median/MAD
#         min_periods = max(3, window // 2)

#     med = x.rolling(window=window, center=center, min_periods=min_periods).median()
#     abs_dev = (x - med).abs()
#     mad = abs_dev.rolling(window=window, center=center, min_periods=min_periods).median()

#     mad_eff = mad.clip(lower=mad_floor)
#     score = abs_dev / mad_eff

#     is_outlier = score > k
#     return is_outlier, med, mad, score


# def hampel_filter_df(
#     df: pd.DataFrame,
#     *,
#     cols: list[str],
#     window: int,
#     k: float,
#     replace: str = "nan",   # "nan" or "median"
#     center: bool = True,
#     min_periods: int | None = None,
#     mad_floor: float = 1e-9,
# ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
#     """
#     Apply Hampel filter to multiple columns feature-by-feature.

#     Returns:
#       cleaned_df
#       outlier_mask_df (bool)
#       summary_df (per column stats)
#     """
#     d = df.copy()
#     out_mask = pd.DataFrame(False, index=d.index, columns=[c for c in cols if c in d.columns])

#     summary_rows = []
#     for c in out_mask.columns:
#         is_out, med, mad, score = hampel_filter_series(
#             d[c],
#             window=window,
#             k=k,
#             center=center,
#             min_periods=min_periods,
#             mad_floor=mad_floor,
#         )
#         out_mask[c] = is_out.fillna(False)

#         n_total = int(d[c].shape[0])
#         n_out = int(out_mask[c].sum())
#         frac = float(n_out / n_total) if n_total else 0.0

#         summary_rows.append({
#             "column": c,
#             "window": int(window),
#             "k": float(k),
#             "outliers": n_out,
#             "rows": n_total,
#             "outlier_frac": frac,
#         })

#         if replace == "nan":
#             d.loc[out_mask[c], c] = np.nan
#         elif replace == "median":
#             d.loc[out_mask[c], c] = med.loc[out_mask[c]]
#         else:
#             raise ValueError("replace must be 'nan' or 'median'")

#     summary = pd.DataFrame(summary_rows).sort_values("outlier_frac", ascending=False).reset_index(drop=True)
#     return d, out_mask, summary

In [ ]:

# weather_cols = [
#     "PV_WS_Radiation",
#     "PV_WS_AirTemp",
#     "PV_WS_RelHum",
#     "PV_WS_RelAirPre",
# ]

# soc_cols = ["BA_Soc"]  # optional

# # --- Apply Hampel in stages (recommended: after comm-loss cleaning) ---
# df_hampel = df_clean.copy()   # IMPORTANT: apply after your comm-loss/maintenance pipeline output


# df_hampel, mask_weather, sum_weather = hampel_filter_df(
#     df_hampel,
#     cols=weather_cols,
#     window=36,
#     k=10,
#     replace="nan",
# )

# df_hampel, mask_soc, sum_soc = hampel_filter_df(
#     df_hampel,
#     cols=soc_cols,
#     window=11,
#     k=4,
#     replace="nan",
# )

# summary_all = pd.concat([
#     sum_weather.assign(category="weather"),
#     sum_soc.assign(category="soc"),
# ], ignore_index=True).sort_values(["category","outlier_frac"], ascending=[True, False])

# print(summary_all)

In [ ]:
# weather_cols = [
#     "PV_WS_Radiation",
#     "PV_WS_AirTemp",
#     "PV_WS_RelHum",
#     "PV_WS_RelAirPre",
# ]

# df_weather_spike_clean, weather_spike_mask, weather_spike_summary = neighbor_spike_filter_df(
#     df_clean,               # after comm-loss cleaning
#     cols=weather_cols,
#     auto_thresholds=True,   # per-column adaptive
#     q_jump=0.999,          # stricter than power (fewer flags)
#     q_neighbor=0.85,        # require neighbors to be VERY consistent
#     detect_1pt=True,
#     detect_2pt=True,
#     replace="nan",
# )

# print(weather_spike_summary)

In [ ]:
import plotly.graph_objects as go

def plot_before_after(df_before, df_after, col, start=None, end=None):
    d1 = df_before.copy()
    d2 = df_after.copy()
    d1.index = pd.to_datetime(d1.index, errors="coerce")
    d2.index = pd.to_datetime(d2.index, errors="coerce")
    d1 = d1[~d1.index.isna()].sort_index()
    d2 = d2[~d2.index.isna()].sort_index()
    if start or end:
        d1 = d1.loc[start:end]
        d2 = d2.loc[start:end]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d1.index, y=pd.to_numeric(d1[col], errors="coerce"),
                             mode="lines", name="Before"))
    fig.add_trace(go.Scatter(x=d2.index, y=pd.to_numeric(d2[col], errors="coerce"),
                             mode="lines", name="After", connectgaps=False))
    fig.update_layout(title=f"Before vs After (Neighbor Spike Clean): {col}",
                      xaxis_title="Time", yaxis_title=col)
    fig.show()

def plot_spikes(df_before, spike_mask, col, start=None, end=None):
    d = df_before.copy()
    d.index = pd.to_datetime(d.index, errors="coerce")
    d = d[~d.index.isna()].sort_index()
    m = spike_mask[col].reindex(d.index).fillna(False)
    if start or end:
        d = d.loc[start:end]
        m = m.loc[start:end]

    y = pd.to_numeric(d[col], errors="coerce")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d.index, y=y, mode="lines", name="Original"))
    fig.add_trace(go.Scatter(x=d.index[m], y=y[m], mode="markers", name="Detected spikes"))
    fig.update_layout(title=f"Detected spikes: {col}",
                      xaxis_title="Time", yaxis_title=col)
    fig.show()

In [ ]:
# plot_before_after(df_clean, df_weather_spike_clean, "PV_WS_Radiation")
# plot_spikes(df_clean, weather_spike_mask, "PV_WS_Radiation")

In [ ]:
# import plotly.graph_objects as go

# def plot_before_after_hampel(df_before, df_after, col, start=None, end=None):
#     d1 = df_before.copy()
#     d2 = df_after.copy()

#     d1.index = pd.to_datetime(d1.index, errors="coerce")
#     d2.index = pd.to_datetime(d2.index, errors="coerce")
#     d1 = d1[~d1.index.isna()].sort_index()
#     d2 = d2[~d2.index.isna()].sort_index()

#     if start or end:
#         d1 = d1.loc[start:end]
#         d2 = d2.loc[start:end]

#     y1 = pd.to_numeric(d1[col], errors="coerce")
#     y2 = pd.to_numeric(d2[col], errors="coerce")

#     fig = go.Figure()

#     fig.add_trace(go.Scatter(
#         x=d1.index, y=y1,
#         mode="lines",
#         name="Before (df_clean)"
#     ))

#     fig.add_trace(go.Scatter(
#         x=d2.index, y=y2,
#         mode="lines",
#         name="After (df_hampel)",
#         connectgaps=False  # show gaps where NaNs are
#     ))

#     fig.update_layout(
#         title=f"Before vs After Hampel: {col}",
#         xaxis_title="Time",
#         yaxis_title=col,
#         legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
#     )
#     fig.show()

In [ ]:
# def plot_hampel_outliers(df_before, outlier_mask, col, start=None, end=None):
#     d = df_before.copy()
#     d.index = pd.to_datetime(d.index, errors="coerce")
#     d = d[~d.index.isna()].sort_index()

#     m = outlier_mask[col].reindex(d.index).fillna(False)

#     if start or end:
#         d = d.loc[start:end]
#         m = m.loc[start:end]

#     y = pd.to_numeric(d[col], errors="coerce")

#     fig = go.Figure()

#     fig.add_trace(go.Scatter(
#         x=d.index, y=y,
#         mode="lines",
#         name="Original (before Hampel)"
#     ))

#     fig.add_trace(go.Scatter(
#         x=d.index[m],
#         y=y[m],
#         mode="markers",
#         name="Hampel outliers",
#     ))

#     fig.update_layout(
#         title=f"Hampel flagged outliers: {col}",
#         xaxis_title="Time",
#         yaxis_title=col,
#         legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
#     )
#     fig.show()

In [ ]:
# def plot_first_outlier_zoom(df_before, df_after, outlier_mask, col, window_hours=6):
#     d = df_before.copy()
#     d.index = pd.to_datetime(d.index, errors="coerce")
#     d = d[~d.index.isna()].sort_index()

#     m = outlier_mask[col].reindex(d.index).fillna(False)
#     if not m.any():
#         print(f"No outliers flagged for {col}.")
#         return

#     t0 = m[m].index[0]
#     start = t0 - pd.Timedelta(hours=window_hours)
#     end   = t0 + pd.Timedelta(hours=window_hours)

#     plot_before_after_hampel(df_before, df_after, col, start=start, end=end)
#     plot_hampel_outliers(df_before, outlier_mask, col, start=start, end=end)

In [ ]:
# weather_cols = ["PV_WS_Radiation","PV_WS_AirTemp","PV_WS_RelHum","PV_WS_RelAirPre"]
# soc_cols = ["BA_Soc"]

# # Plot zoom around first flagged outlier for each column
# for c in weather_cols:
#     plot_first_outlier_zoom(df_clean, df_hampel, mask_weather, c, window_hours=6)

# for c in soc_cols:
#     plot_first_outlier_zoom(df_clean, df_hampel, mask_soc, c, window_hours=6)

In [ ]:
# import plotly.graph_objects as go
# import pandas as pd

# def plot_before_after(df_before, df_after, col, start=None, end=None, title=None):
#     d1 = df_before.copy()
#     d2 = df_after.copy()

#     d1.index = pd.to_datetime(d1.index, errors="coerce")
#     d2.index = pd.to_datetime(d2.index, errors="coerce")
#     d1 = d1[~d1.index.isna()].sort_index()
#     d2 = d2[~d2.index.isna()].sort_index()

#     if start or end:
#         d1 = d1.loc[start:end]
#         d2 = d2.loc[start:end]

#     fig = go.Figure()

#     fig.add_trace(go.Scatter(
#         x=d1.index, y=pd.to_numeric(d1[col], errors="coerce"),
#         mode="lines", name=f"Before (df_clean)"
#     ))

#     fig.add_trace(go.Scatter(
#         x=d2.index, y=pd.to_numeric(d2[col], errors="coerce"),
#         mode="lines", name=f"After (df_spike_clean)",
#         connectgaps=False  # IMPORTANT: shows gaps where NaNs are
#     ))

#     fig.update_layout(
#         title=title or f"Before vs After Spike Cleaning: {col}",
#         xaxis_title="Time",
#         yaxis_title=col,
#         legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
#     )
#     fig.show()

# # Example
# plot_before_after(
#     df_clean,
#     df_spike_clean,
#     col="BU_TotActPwr_Academy",
# )

In [ ]:
# -----------------------------
# Small helpers
# -----------------------------

def _max_consecutive_true(mask: pd.Series) -> int:
    """Return max run length of consecutive True values (mask aligned to index)."""
    m = mask.fillna(False).astype(bool)
    if not m.any():
        return 0
    g = (m != m.shift()).cumsum()
    # lengths of True groups only
    lens = m.groupby(g).sum()
    # But sum works because True=1, False=0 and groups are constant
    # Filter to groups that are True
    first_vals = m.groupby(g).first()
    true_groups = lens[first_vals]
    return int(true_groups.max()) if len(true_groups) else 0


def _coerce_numeric_inplace(d: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Coerce selected columns to numeric once (speeds up downstream)."""
    for c in cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")
    return d


# -----------------------------
# Group detectors (raw intervals)
# -----------------------------

def detect_bess_limit_intervals_raw(
    df: pd.DataFrame,
    *,
    freq: str = "5min",
    tol_zero_limits: float = 0.0,
    min_minutes: int = 30,
    chr_col: str = "BA_PwrAtChrLimTot",
    dis_col: str = "BA_PwrAtDisLimTot",
):
    """
    Raw BESS candidate intervals:
      (ChrLim ~ 0) & (DisLim ~ 0) for >= min_minutes
    """
    d = ensure_datetime_index(df)

    for c in (chr_col, dis_col):
        if c not in d.columns:
            raise KeyError(f"Missing required column: {c}")

    # NOTE: use abs() so negatives don't accidentally count as "zero"
    chr_v = pd.to_numeric(d[chr_col], errors="coerce")
    dis_v = pd.to_numeric(d[dis_col], errors="coerce")
    base_mask = (chr_v.abs() <= tol_zero_limits) & (dis_v.abs() <= tol_zero_limits)

    step = pd.to_timedelta(freq)
    min_samples = int(np.ceil(pd.Timedelta(minutes=min_minutes) / step))

    event_mask = keep_long_runs(base_mask, min_samples=min_samples)
    intervals_raw = run_intervals(event_mask)

    debug = {
        "group": "BESS",
        "event_type": "bess_limits_zero",
        "total_rows": int(len(d)),
        "base_hits": int(base_mask.sum()),
        "event_hits_after_duration_filter": int(event_mask.sum()),
        "num_intervals_raw": int(len(intervals_raw)),
        "min_samples": int(min_samples),
        "freq": freq,
        "tol_zero_limits": float(tol_zero_limits),
        "min_minutes": int(min_minutes),
    }
    return d, intervals_raw, debug


def detect_pv_weather_zero_intervals_raw(
    df: pd.DataFrame,
    *,
    pv_cols: list[str],
    freq: str = "5min",
    min_minutes: int = 30,
    zero_tol: float = 0.0,
    min_cols_fraction: float = 0.75,
):
    """
    Raw PV candidate intervals:
      fraction_of_cols_near_zero(t) >= min_cols_fraction for >= min_minutes
    """
    d = ensure_datetime_index(df)

    cols = [c for c in pv_cols if c in d.columns]
    if not cols:
        raise KeyError("None of pv_cols exist in df.")

    step = pd.to_timedelta(freq)
    min_samples = int(np.ceil(pd.Timedelta(minutes=min_minutes) / step))

    # Build per-column near-zero mask
    Z = pd.DataFrame(index=d.index)
    for c in cols:
        x = pd.to_numeric(d[c], errors="coerce")
        Z[c] = (x.abs() <= zero_tol)

    frac_zero = Z.mean(axis=1)
    base_mask = frac_zero >= min_cols_fraction

    event_mask = keep_long_runs(base_mask, min_samples=min_samples)
    intervals_raw = run_intervals(event_mask)

    debug = {
        "group": "PV_WEATHER",
        "event_type": "pv_sensors_zero",
        "total_rows": int(len(d)),
        "pv_cols_used": cols,
        "min_cols_fraction": float(min_cols_fraction),
        "min_minutes": int(min_minutes),
        "freq": freq,
        "min_samples": int(min_samples),
        "base_hits": int(base_mask.sum()),
        "event_hits_after_duration_filter": int(event_mask.sum()),
        "num_intervals_raw": int(len(intervals_raw)),
        "zero_tol": float(zero_tol),
    }
    return d, intervals_raw, debug


# -----------------------------
# Combined pipeline
# -----------------------------

def pv_bess_comm_loss_pipeline(
    df: pd.DataFrame,
    *,
    bess_cols: list[str],
    pv_sensor_cols: list[str],
    # common
    freq: str = "5min",
    merge_gap_minutes: int = 30,

    # BESS detection
    tol_zero_limits: float = 0.0,
    bess_min_minutes: int = 30,
    chr_col: str = "BA_PwrAtChrLimTot",
    dis_col: str = "BA_PwrAtDisLimTot",

    # BESS validation (graded stuck + consecutive zero-run)
    zero_tol_signals: float = 0.0,
    high_thr: float = 0.95,
    med_thr: float = 0.85,
    probable_min_hours: float = 2.0,
    change_rate_thr: float = 0.01,
    change_min_hours: float = 1.0,
    min_non_nan: int = 10,
    nanify_levels: tuple[str, ...] = ("definite", "probable"),

    # Extra: consecutive zero-run check inside interval (requested)
    # If a signal has a near-zero run >= this many minutes inside the interval,
    # we can upgrade it to "probable" (or "definite" if you want via thresholds).
    zero_run_min_minutes: int = 60,

    # PV detection & cleaning
    pv_min_minutes: int = 30,
    pv_zero_tol: float = 0.0,
    pv_min_cols_fraction: float = 0.75,
):
    """
    Combined PV + BESS comm-loss/maintenance pipeline.

    Key property:
      PV and BESS intervals are detected/merged/cleaned independently.
      They can overlap or be totally different lengths/times — that’s expected.

    Returns:
      df_clean
      events_all (merged intervals from both groups)
      bess_report (interval x column validation metrics)
      debug (dict with pv_debug + bess_debug)
    """
    d0 = ensure_datetime_index(df).copy()

    # Convert numeric once for relevant cols (big speed-up)
    numeric_cols = sorted(set([c for c in bess_cols + pv_sensor_cols + [chr_col, dis_col] if c in d0.columns]))
    d0 = _coerce_numeric_inplace(d0, numeric_cols)

    step = pd.to_timedelta(freq)
    merge_gap = pd.Timedelta(minutes=int(merge_gap_minutes))

    # ---- Detect RAW intervals ----
    d_bess, bess_raw, bess_debug = detect_bess_limit_intervals_raw(
        d0,
        freq=freq,
        tol_zero_limits=tol_zero_limits,
        min_minutes=bess_min_minutes,
        chr_col=chr_col,
        dis_col=dis_col,
    )

    d_pv, pv_raw, pv_debug = detect_pv_weather_zero_intervals_raw(
        d0,
        pv_cols=pv_sensor_cols,
        freq=freq,
        min_minutes=pv_min_minutes,
        zero_tol=pv_zero_tol,
        min_cols_fraction=pv_min_cols_fraction,
    )

    # ---- Merge BEFORE cleaning (your preference) ----
    bess_merged = merge_intervals(bess_raw, max_gap=merge_gap)
    pv_merged = merge_intervals(pv_raw, max_gap=merge_gap)

    # ---- Cleaning ----
    cleaned = d0.copy()

    # PV: directly NaN weather sensor cols in merged intervals
    pv_cols_existing = [c for c in pv_sensor_cols if c in cleaned.columns]
    for _, r in pv_merged.iterrows():
        a, b = r["start"], r["end"]
        cleaned.loc[a:b, pv_cols_existing] = np.nan

    # BESS: graded validation per signal inside merged intervals + optional zero-run escalation
    bess_cols_existing = [c for c in bess_cols if c in cleaned.columns]
    rows = []

    # how many samples correspond to zero_run_min_minutes
    zero_run_min_samples = int(np.ceil(pd.Timedelta(minutes=int(zero_run_min_minutes)) / step))

    for interval_id, r in bess_merged.reset_index(drop=True).iterrows():
        a, b = r["start"], r["end"]

        for col in bess_cols_existing:
            seg = cleaned.loc[a:b, col]  # seg already numeric (coerced once)

            stuck_level, reason, metrics = stuck_decision_graded(
                seg,
                step=step,
                zero_tol=zero_tol_signals,
                high_thr=high_thr,
                med_thr=med_thr,
                probable_min_hours=probable_min_hours,
                change_rate_thr=change_rate_thr,
                change_min_hours=change_min_hours,
                min_non_nan=min_non_nan,
            )

            # Extra: consecutive near-zero run check (requested)
            # If we see a long enough near-zero run, upgrade "no" -> "probable"
            # (you can later tighten/loosen this rule by signal type)
            near_zero_mask = pd.to_numeric(seg, errors="coerce").abs() <= zero_tol_signals
            max_zero_run = _max_consecutive_true(near_zero_mask)
            if (stuck_level == "no") and (max_zero_run >= zero_run_min_samples):
                stuck_level = "probable"
                reason = "long_consecutive_zero_run"

            rows.append({
                "group": "BESS",
                "event_type": "bess_limits_zero",
                "interval_id": int(interval_id),
                "start": a,
                "end": b,
                "column": col,
                "stuck_level": stuck_level,
                "reason": reason,
                "max_consecutive_zero_samples": int(max_zero_run),
                **metrics,
            })

            if stuck_level in nanify_levels:
                cleaned.loc[a:b, col] = np.nan

    bess_report = pd.DataFrame(rows)

    # ---- Events table (both groups, merged, independent) ----
    def _events_df(intervals: pd.DataFrame, group: str, event_type: str) -> pd.DataFrame:
        if intervals is None or intervals.empty:
            return pd.DataFrame(columns=["group","event_type","start","end","minutes"])
        out = intervals.copy()
        out["group"] = group
        out["event_type"] = event_type
        # minutes (approx using time diff)
        out["minutes"] = ((out["end"] - out["start"]) / pd.Timedelta(minutes=1)).astype(float)
        out = out.loc[:, ["group","event_type","start","end","minutes"]].sort_values(["group","start"]).reset_index(drop=True)
        out.insert(0, "event_id", np.arange(1, len(out) + 1))
        return out

    events_all = pd.concat([
        _events_df(bess_merged, "BESS", "bess_limits_zero"),
        _events_df(pv_merged, "PV_WEATHER", "pv_sensors_zero"),
    ], ignore_index=True).sort_values(["start","group"]).reset_index(drop=True)

    debug = {
        "freq": freq,
        "merge_gap_minutes": int(merge_gap_minutes),
        "bess_debug": {
            **bess_debug,
            "num_intervals_merged": int(len(bess_merged)),
            "nanify_levels": list(nanify_levels),
            "zero_run_min_minutes": int(zero_run_min_minutes),
            "zero_run_min_samples": int(zero_run_min_samples),
        },
        "pv_debug": {
            **pv_debug,
            "num_intervals_merged": int(len(pv_merged)),
            "nanified_cols": pv_cols_existing,
        },
        "events_total": int(len(events_all)),
    }

    return cleaned, events_all, bess_report, debug


In [ ]:

# -----------------------------
# Example usage (your groups)
# -----------------------------
cleaned_df, events_all, bess_report, debug = pv_bess_comm_loss_pipeline(
    df,
    bess_cols=GROUPS["BESS"],
    pv_sensor_cols=[
        "PV_WS_Radiation", "PV_WS_AirTemp", "PV_WS_RelHum", "PV_WS_RelAirPre"
    ],
    freq="5min",
    merge_gap_minutes=30,
    # BESS detector
    tol_zero_limits=0.0,
    bess_min_minutes=30,
    # BESS validator
    zero_tol_signals=0.0,
    high_thr=0.95,
    med_thr=0.85,
    probable_min_hours=2.0,
    change_rate_thr=0.01,
    change_min_hours=1.0,
    min_non_nan=10,
    nanify_levels=("definite","probable"),
    zero_run_min_minutes=60,
    # PV detector
    pv_min_minutes=30,
    pv_zero_tol=0.0,
    pv_min_cols_fraction=0.75,
)

print(debug)
print(events_all.to_string(index=False))
print(bess_report.head(20).to_string(index=False))

### neighbor based spike

In [ ]:
# ============================================================
# Optimized neighbor-based spike detection (1pt + 2pt)
# - safer (datetime interpolation checks)
# - fewer pandas objects created per column
# - fixed 2pt "outer_close_thr" (no hidden *2)
# - always absolute diffs (spike detection should be sign-agnostic)
# ============================================================

def neighbor_spike_mask_1pt(
    s: pd.Series,
    *,
    jump_thr: float,
    neighbor_close_thr: float,
) -> pd.Series:
    """
    1-sample isolated spike at t if:
      |x[t-1] - x[t+1]| <= neighbor_close_thr
      and |x[t] - x[t-1]| >= jump_thr
      and |x[t] - x[t+1]| >= jump_thr
    """
    x = pd.to_numeric(s, errors="coerce")
    prev = x.shift(1)
    nxt  = x.shift(-1)

    ok_neighbors = (prev - nxt).abs().le(neighbor_close_thr)
    big_jump = (x - prev).abs().ge(jump_thr) & (x - nxt).abs().ge(jump_thr)

    mask = ok_neighbors & big_jump
    mask = mask & x.notna() & prev.notna() & nxt.notna()
    return mask.fillna(False)


def neighbor_spike_mask_2pt(
    s: pd.Series,
    *,
    jump_thr: float,
    outer_close_thr: float,
) -> pd.Series:
    """
    2-sample isolated glitch at t,t+1 if:
      |x[t-1] - x[t+2]| <= outer_close_thr
      and both inner points are far from both outer points by >= jump_thr

    Marks BOTH inner points True.
    """
    x = pd.to_numeric(s, errors="coerce")
    xm1 = x.shift(1)     # t-1
    x0  = x              # t
    xp1 = x.shift(-1)    # t+1
    xp2 = x.shift(-2)    # t+2

    ok_outer = (xm1 - xp2).abs().le(outer_close_thr)

    big_jump_pair = (
        (x0  - xm1).abs().ge(jump_thr) & (x0  - xp2).abs().ge(jump_thr) &
        (xp1 - xm1).abs().ge(jump_thr) & (xp1 - xp2).abs().ge(jump_thr)
    )

    core = ok_outer & big_jump_pair
    core = core & xm1.notna() & x0.notna() & xp1.notna() & xp2.notna()
    core = core.fillna(False)

    # mark both inner points (t and t+1)
    mask = core | core.shift(1, fill_value=False)
    return mask.fillna(False)


def estimate_thresholds_from_ramps(
    s: pd.Series,
    *,
    q_jump: float = 0.999,
    q_neighbor: float = 0.90,
    min_jump: float = 0.0,
) -> tuple[float, float]:
    """
    jump_thr: high quantile of |diff| (typical max ramp)
    neighbor_close_thr: quantile of |x[t-1]-x[t+1]| (typical neighbor variation)
    """
    x = pd.to_numeric(s, errors="coerce")
    ramps = x.diff().abs().dropna()
    if ramps.empty:
        return (float(min_jump), 0.0)

    jump_thr = float(ramps.quantile(q_jump))
    jump_thr = max(jump_thr, float(min_jump))

    neigh = (x.shift(1) - x.shift(-1)).abs().dropna()
    neighbor_thr = float(neigh.quantile(q_neighbor)) if not neigh.empty else 0.0

    return jump_thr, neighbor_thr


def neighbor_spike_filter_df(
    df: pd.DataFrame,
    *,
    cols: list[str],

    # thresholds (global or per-column)
    auto_thresholds: bool = True,
    q_jump: float = 0.999,
    q_neighbor: float = 0.90,
    min_jump: float = 0.0,

    # if auto_thresholds=False:
    jump_thr: float | None = None,
    neighbor_close_thr: float | None = None,
    outer_close_thr: float | None = None,

    # which glitches:
    detect_1pt: bool = True,
    detect_2pt: bool = True,

    # replacement:
    replace: str = "nan",   # "nan" or "interp"

    # interpolation behavior:
    interp_method: str = "time",  # "time" or "linear"
    interp_limit_direction: str = "both",  # "forward"|"backward"|"both"
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Apply neighbor-based spike detection feature-by-feature.

    Returns:
      cleaned_df
      spike_mask_df (bool)
      summary_df (spikes per column, thresholds used)

    Notes:
      - If replace="interp" and interp_method="time", index MUST be a DatetimeIndex.
      - This is best applied AFTER comm-loss/maintenance NaN pipeline.
    """
    d = df.copy()

    # Prepare index for time interpolation if needed
    if replace == "interp" and interp_method == "time":
        if not isinstance(d.index, pd.DatetimeIndex):
            d.index = pd.to_datetime(d.index, errors="coerce")
        d = d[~d.index.isna()].sort_index()
        if not isinstance(d.index, pd.DatetimeIndex):
            raise ValueError("For interp_method='time', df.index must be a valid DatetimeIndex.")

    used_cols = [c for c in cols if c in d.columns]
    mask_df = pd.DataFrame(False, index=d.index, columns=used_cols)

    rows = []
    for c in used_cols:
        x = pd.to_numeric(d[c], errors="coerce")

        if auto_thresholds:
            jt, nt = estimate_thresholds_from_ramps(
                x, q_jump=q_jump, q_neighbor=q_neighbor, min_jump=min_jump
            )
            ot = nt if outer_close_thr is None else float(outer_close_thr)
        else:
            if jump_thr is None or neighbor_close_thr is None:
                raise ValueError("If auto_thresholds=False, provide jump_thr and neighbor_close_thr.")
            jt = float(jump_thr)
            nt = float(neighbor_close_thr)
            ot = float(outer_close_thr) if outer_close_thr is not None else nt

        m = pd.Series(False, index=d.index)
        if detect_1pt:
            m |= neighbor_spike_mask_1pt(x, jump_thr=jt, neighbor_close_thr=nt)
        if detect_2pt:
            m |= neighbor_spike_mask_2pt(x, jump_thr=jt, outer_close_thr=ot)

        m = m.fillna(False)
        mask_df[c] = m

        n_total = int(len(x))
        n_spike = int(m.sum())

        rows.append({
            "column": c,
            "rows": n_total,
            "spikes": n_spike,
            "spike_frac": (n_spike / n_total) if n_total else 0.0,
            "jump_thr": jt,
            "neighbor_close_thr": nt,
            "outer_close_thr": ot,
            "auto_thresholds": bool(auto_thresholds),
            "detect_1pt": bool(detect_1pt),
            "detect_2pt": bool(detect_2pt),
            "replace": replace,
            "interp_method": interp_method if replace == "interp" else None,
        })

        # Apply replacement
        if replace == "nan":
            d.loc[m, c] = np.nan

        elif replace == "interp":
            # NaN-out spikes first
            d.loc[m, c] = np.nan

            # interpolate
            if interp_method == "time":
                d[c] = d[c].interpolate(method="time", limit_direction=interp_limit_direction)
            elif interp_method == "linear":
                d[c] = d[c].interpolate(method="linear", limit_direction=interp_limit_direction)
            else:
                raise ValueError("interp_method must be 'time' or 'linear'")

            # fallback edges
            d[c] = d[c].ffill().bfill()

        else:
            raise ValueError("replace must be 'nan' or 'interp'")

    summary = pd.DataFrame(rows).sort_values("spike_frac", ascending=False).reset_index(drop=True)
    return d, mask_df, summary

In [ ]:
# df_before_spike = df_clean.copy()

# spike_cols = [
#     # PV weather (good spike candidates)
#     "PV_WS_Radiation", "PV_WS_AirTemp", "PV_WS_RelHum",
#     # BESS (if you use them as ML inputs)
#     "BA_Soc","BA_TotActPwr_BESS_AC_Panel2",
#     "BA_TotActPwr_BESS_AC_Panel1",
#     # Building
#     "BU_TotActPwr_Tech_Room", "BU_TotActPwr_SDB_EL_Substation",
#     "BU_TotActPwr_Academy", "BU_PwrReq",
#     # ELX (only if needed)
#     "EL_TotActPwr_NitrogenUnit",
# ]
# spike_cols = [c for c in spike_cols if c in df_before_spike.columns]

# df_after_spike, spike_mask, spike_summary = neighbor_spike_filter_df(
#     df_before_spike,
#     cols=spike_cols,          # ✅ REQUIRED
#     auto_thresholds=True,
#     q_jump=0.995,
#     q_neighbor=0.90,
#     detect_1pt=True,
#     detect_2pt=True,
#     replace="nan",
# )

# spike_table = spike_summary_table(df_before_spike, df_after_spike, spike_mask, spike_summary)
# print(spike_table.head(30).to_string(index=False))

In [ ]:
def spike_summary_table(
    df_before_spike: pd.DataFrame,
    df_after_spike: pd.DataFrame,
    spike_mask_df: pd.DataFrame,
    spike_summary: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """
    Build a clean thesis-friendly summary table for spike detection.

    Shows (per column):
      - total rows
      - non-NaN before
      - spikes detected
      - spikes as % of rows
      - spikes as % of available (non-NaN) before
      - new NaNs introduced by spike stage (should match spikes if replace="nan")
      - thresholds (if provided in spike_summary)

    Returns a DataFrame.
    """
    cols = [c for c in spike_mask_df.columns if c in df_before_spike.columns and c in df_after_spike.columns]

    rows = []
    for c in cols:
        raw = pd.to_numeric(df_before_spike[c], errors="coerce")
        clean = pd.to_numeric(df_after_spike[c], errors="coerce")
        m = spike_mask_df[c].fillna(False).astype(bool)

        n_rows = int(len(raw))
        n_non_nan_before = int(raw.notna().sum())
        n_spikes = int(m.sum())

        # new NaNs introduced by spike filter (raw had value, now NaN)
        new_nans = int((raw.notna() & clean.isna()).sum())

        rows.append({
            "column": c,
            "rows": n_rows,
            "non_nan_before": n_non_nan_before,
            "spikes_detected": n_spikes,
            "spikes_%_of_rows": (n_spikes / n_rows * 100.0) if n_rows else np.nan,
            "spikes_%_of_non_nan": (n_spikes / n_non_nan_before * 100.0) if n_non_nan_before else np.nan,
            "new_nans_introduced": new_nans,
        })

    out = pd.DataFrame(rows)

    # attach thresholds from spike_summary (if available)
    if spike_summary is not None and not spike_summary.empty and "column" in spike_summary.columns:
        keep_cols = [c for c in [
            "column", "jump_thr", "neighbor_close_thr", "outer_close_thr",
            "auto_thresholds", "detect_1pt", "detect_2pt", "replace", "interp_method"
        ] if c in spike_summary.columns]
        out = out.merge(spike_summary[keep_cols], on="column", how="left")

    # nice ordering
    sort_key = "spikes_%_of_non_nan" if "spikes_%_of_non_nan" in out.columns else "spikes_detected"
    out = out.sort_values(sort_key, ascending=False).reset_index(drop=True)
    return out


In [ ]:
import plotly.graph_objects as go


def plot_spikes_raw_vs_clean(
    df_raw: pd.DataFrame,
    df_clean: pd.DataFrame,
    spike_mask_df: pd.DataFrame,
    col: str,
    *,
    start=None,
    end=None,
    title: str | None = None,
    show_cleaned: bool = True,
    show_spike_markers: bool = True,
    marker_symbol: str = "x",
    marker_size: int = 9,
):

    if col not in df_raw.columns:
        raise KeyError(f"'{col}' not in df_raw")
    if col not in df_clean.columns:
        raise KeyError(f"'{col}' not in df_clean")
    if col not in spike_mask_df.columns:
        raise KeyError(f"'{col}' not in spike_mask_df")

    d0 = df_raw.copy()
    d1 = df_clean.copy()
    mdf = spike_mask_df.copy()

    # align + ensure datetime-like
    d0.index = pd.to_datetime(d0.index, errors="coerce")
    d1.index = pd.to_datetime(d1.index, errors="coerce")
    mdf.index = pd.to_datetime(mdf.index, errors="coerce")

    d0 = d0[~d0.index.isna()].sort_index()
    d1 = d1[~d1.index.isna()].sort_index()
    mdf = mdf[~mdf.index.isna()].sort_index()

    # slice
    if start is not None or end is not None:
        d0 = d0.loc[start:end]
        d1 = d1.loc[start:end]
        mdf = mdf.loc[start:end]

    # align indexes (intersection)
    idx = d0.index.intersection(d1.index).intersection(mdf.index)
    d0 = d0.loc[idx]
    d1 = d1.loc[idx]
    m = mdf.loc[idx, col].fillna(False).astype(bool)

    raw = pd.to_numeric(d0[col], errors="coerce")
    clean = pd.to_numeric(d1[col], errors="coerce")

    fig = go.Figure()

    # raw
    fig.add_trace(go.Scatter(
        x=raw.index, y=raw.values,
        mode="lines",
        name="raw"
    ))

    # cleaned
    if show_cleaned:
        fig.add_trace(go.Scatter(
            x=clean.index, y=clean.values,
            mode="lines",
            name="cleaned"
        ))

    # spikes markers: show raw values at spike timestamps
    if show_spike_markers and m.any():
        fig.add_trace(go.Scatter(
            x=raw.index[m],
            y=raw[m],
            mode="markers",
            name="spike points (raw value)",
            marker=dict(symbol=marker_symbol, size=marker_size)
        ))

    fig.update_layout(
        title=title or f"{col}: raw vs cleaned + spike markers",
        xaxis_title="Time",
        yaxis_title=col,
        legend=dict(orientation="h")
    )
    fig.show()


def plot_spikes_for_columns(
    df_raw: pd.DataFrame,
    df_clean: pd.DataFrame,
    spike_mask_df: pd.DataFrame,
    cols: list[str],
    *,
    start=None,
    end=None,
):
    """
    Convenience wrapper: loops through cols and plots one by one.
    """
    for c in cols:
        if c in df_raw.columns and c in df_clean.columns and c in spike_mask_df.columns:
            plot_spikes_raw_vs_clean(
                df_raw, df_clean, spike_mask_df, c,
                start=start, end=end
            )

In [ ]:
# # 1) keep a raw copy before spike filtering
# df_before_spike = df_clean.copy()

# # 2) run spike filter (example)
# df_after_spike, spike_mask, spike_summary = neighbor_spike_filter_df(
#     df_before_spike,
#     cols=pv_sensor_cols,          # or your full list
#     auto_thresholds=True,
#     q_jump=0.995,
#     q_neighbor=0.90,
#     detect_1pt=True,
#     detect_2pt=True,
#     replace="nan",
# )

# # 3) plot one column with spikes highlighted (raw value markers)
# plot_spikes_raw_vs_clean(
#     df_before_spike,      # raw for spike-stage
#     df_after_spike,       # cleaned after spike removal
#     spike_mask,
#     "BU_TotActPwr_Tech_Room",
# )

In [ ]:
GROUPS = {
    "Building_load": [
        "BU_TotActPwr_Tech_Room",
        "ELX_TotActPwr_Pump_Room",
        "BU_TotActPwr_SDB_EL_Substation",
        "BU_TotActPwr_Academy",
        "BU_PwrReq",
        "BU_Unitstate"

    ],
    "pv_weather": [
        "PV_WS_Radiation",
        "PV_WS_AirTemp",
        "PV_WS_RelHum",
        "PV_WS_RelAirPre",
        "PV_Unitstate"
    ],
    "BESS": [
        "BA_PwrAtChrLimTot",
        "BA_PwrAtDisLimTot",
        "BA_TotActPwr_BESS_AC_Panel2",
        "BA_TotActPwr_BESS_AC_Panel1",
        "BA_PwrAt",
        "BA_Unitstate",
        "BA_Soc"
    ],
    "ELX": [
        'EL_AuxPwr', 'EL_TotActPwr_NitrogenUnit'
    ],
}